In [1]:
START_DATE = "01/2025"
END_DATE = "06/2025"

In [2]:
import datetime
from pathlib import Path

from nwec.utility_reporting.analysis import compile_report

In [3]:
start_dt = datetime.datetime.strptime(START_DATE, "%m/%Y").replace(tzinfo=datetime.UTC)
end_dt = datetime.datetime.strptime(END_DATE, "%m/%Y").replace(tzinfo=datetime.UTC)

In [4]:
report_path = compile_report.copy_template(start_dt, end_dt)

In [ ]:
import datetime

from pptx import Presentation

from nwec.constants import CLEAN_UTILITY_DATA

start_month = start_dt.strftime("%B")
end_month = end_dt.strftime("%B")
start_year = start_dt.year
end_year = end_dt.year
date_range = ""

if start_year != end_year:
    date_range = f"{start_month} {start_year} - {end_month} {end_year}"
else:
    date_range = f"{start_month} - {end_month} {start_year}"
# Open the presentation
prs = Presentation(report_path)

# Get the slides
title_slide = prs.slides[0]
arrearage_counts_slide = prs.slides[4]  # Fifth slide (0-indexed)
arrearage_amounts_slide = prs.slides[5]
residential_arrearage_vintage_slide = prs.slides[6]
kli_arrearage_vintage_slide = prs.slides[7]

for shape in title_slide.shapes:
    if hasattr(shape, "text_frame"):
        for paragraph in shape.text_frame.paragraphs:  # type: ignore[reportAttributeAccessIssue]
            for run in paragraph.runs:
                if "{DATE_RANGE}" in run.text:
                    run.text = run.text.replace("{DATE_RANGE}", date_range)

compile_report.add_centered_image(
    prs, arrearage_counts_slide, Path(CLEAN_UTILITY_DATA / "arrearage_counts_stacked.png")
)
prs.save(report_path)